# fastfuncstuff — Colab Quick Test

Motion correction + ICA on a single run from Google Drive.

**Setup:** Update the paths in the first code cell, then Run All.

In [ ]:
# ============================================================
# CONFIGURE THESE
# ============================================================

# Path to your fastfuncstuff repo on Google Drive
FFS_REPO = "/content/drive/MyDrive/code/fastfuncstuff"

# Path to a single 4D NIfTI on Google Drive
INPUT_FILE = "/content/drive/MyDrive/data/my_epi.nii.gz"

# Output directory (local disk is faster than Drive for intermediate I/O)
OUT_DIR = "/content/ffs_output"

## 1. Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# System deps: pigz (fast gzip), zstd (nii.zst support)
!apt-get -qq install pigz zstd

In [ ]:
# Install fastfuncstuff from your Google Drive copy
!pip install -q -e "{FFS_REPO}"

In [ ]:
# Verify install
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

import fastfuncstuff
print(f"fastfuncstuff imported OK")

## 2. Copy input to local disk

Google Drive I/O is slow — copy the file to `/content/` first.

In [ ]:
import shutil, os, time
from pathlib import Path

Path(OUT_DIR).mkdir(exist_ok=True)
local_input = f"/content/{Path(INPUT_FILE).name}"

if not Path(local_input).exists():
    print(f"Copying {INPUT_FILE} to local disk...")
    t0 = time.time()
    shutil.copy2(INPUT_FILE, local_input)
    print(f"  Done in {time.time() - t0:.1f}s  ({Path(local_input).stat().st_size / 1e9:.2f} GB)")
else:
    print(f"Already cached: {local_input}")

## 3. Motion Correction

In [ ]:
moco_prefix = f"{OUT_DIR}/moco"
moco_output = f"{moco_prefix}.nii.gz"
moco_params = f"{moco_prefix}_params.1D"

!ffs_moco \
    -input "{local_input}" \
    -prefix "{moco_output}" \
    -1Dfile "{moco_params}" \
    -twopass \
    -verb 1

In [ ]:
# Quick plot of motion parameters
import numpy as np
import matplotlib.pyplot as plt

if Path(moco_params).exists():
    mp = np.loadtxt(moco_params)
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    labels_rot = ["roll", "pitch", "yaw"]
    labels_trans = ["dS", "dL", "dP"]
    for i in range(3):
        axes[0].plot(mp[:, i], label=labels_rot[i])
        axes[1].plot(mp[:, i + 3], label=labels_trans[i])
    axes[0].set_ylabel("Rotation (deg)")
    axes[0].legend(loc="upper right")
    axes[1].set_ylabel("Translation (mm)")
    axes[1].set_xlabel("Volume")
    axes[1].legend(loc="upper right")
    fig.suptitle("Motion Parameters")
    plt.tight_layout()
    plt.show()
else:
    print("No motion parameter file found — check moco output above.")

## 4. ICA

Run on the motion-corrected output (or on the raw input — change `ica_input` below).

In [ ]:
# Use moco output if available, otherwise fall back to raw input
ica_input = moco_output if Path(moco_output).exists() else local_input
ica_prefix = f"{OUT_DIR}/ica"

!ffs_ica \
    -input "{ica_input}" \
    -prefix "{ica_prefix}" \
    -verbose

In [ ]:
# Show ICA spatial maps (middle slice of first few components)
import nibabel as nib

maps_file = sorted(Path(OUT_DIR).glob("ica_*_ica_maps.nii.gz"))
if maps_file:
    img = nib.load(str(maps_file[0]))
    data = img.get_fdata()
    n_comp = min(data.shape[-1], 12)
    mid_z = data.shape[2] // 2

    cols = 4
    rows = (n_comp + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = axes.flat
    for i in range(n_comp):
        sl = data[:, :, mid_z, i].T
        vmax = max(abs(sl.min()), abs(sl.max())) or 1
        axes[i].imshow(sl, cmap="RdBu_r", vmin=-vmax, vmax=vmax, origin="lower")
        axes[i].set_title(f"IC {i+1}", fontsize=9)
        axes[i].axis("off")
    for j in range(n_comp, len(axes)):
        axes[j].axis("off")
    fig.suptitle(f"ICA spatial maps (z={mid_z})")
    plt.tight_layout()
    plt.show()
else:
    print("No ICA map file found — check ICA output above.")

In [ ]:
# Show ICA timecourses
tc_files = sorted(Path(OUT_DIR).glob("ica_*_ica_timecourses.1D"))
if tc_files:
    tc = np.loadtxt(str(tc_files[0]))
    n_show = min(tc.shape[1], 8)
    fig, axes = plt.subplots(n_show, 1, figsize=(12, 1.5 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for i in range(n_show):
        axes[i].plot(tc[:, i], linewidth=0.7)
        axes[i].set_ylabel(f"IC {i+1}", fontsize=8)
        axes[i].tick_params(labelsize=7)
    axes[-1].set_xlabel("Volume")
    fig.suptitle("ICA Timecourses")
    plt.tight_layout()
    plt.show()
else:
    print("No timecourse file found.")

## 5. Copy results back to Drive (optional)

In [ ]:
# Uncomment to copy results back to Google Drive
# drive_out = "/content/drive/MyDrive/data/ffs_output"
# !mkdir -p "{drive_out}" && cp -v {OUT_DIR}/* "{drive_out}/"